In [1]:
import torch
import torch.nn as nn
import sys
import hls4ml

COMMON_PATH = "../../Common/ESPERTA"
sys.path.append(COMMON_PATH)
try:
	from espertaTorch import MultiEsperta
except ModuleNotFoundError as e:
	print(f"Error: Could not import MMSNet modules. Check COMMON_PATH: {COMMON_PATH}")
	raise e

In [2]:
# Example configuration matching original models
ESPERTA_CONFIGS = [
    # s1 models
    ([-6.07, -1.75, 1.14, 0.56], 0.28),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
    
    # s2 models
    ([-6.07, -1.75, 1.14, 0.56], 0.35),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
]

# Initialize compound model
model = MultiEsperta(ESPERTA_CONFIGS)

In [3]:
config = hls4ml.utils.config_from_pytorch_model(
    model, 
    input_shape=[(1, 512, 512), (1,1)],
    granularity='model', 
    backend='catapult',
    )
print("-----------------------------------")
print("Configuration")
print(config)
print("-----------------------------------")
hls_model = hls4ml.converters.convert_from_pytorch_model(
    model, hls_config=config,
    backend='catapult',
    output_dir='catapult_model/hls4ml_prj',
)

Exception: Unsupported function gt

In [ ]:
hls_model.compile()

In [ ]:
import subprocess

def check_command_exists(command):
    """Check if a command exists in the system PATH"""
    try:
        subprocess.check_output(['which', command], stderr=subprocess.STDOUT)
        return True
    except subprocess.CalledProcessError:
        return False

# Check if catapult command exists
if check_command_exists('catapult'):
    print("Catapult command found. Proceeding with build...")
    hls_model.build(csim=False)
else:
    print("Catapult command not found. Skipping build.")